In [2]:
import os
import glob
import polars as pl
from pathlib import Path

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
# Feature Data Directory (Where the CSVs to be updated are located)

# Change the Drive letter to "F" or "D" depending on the mounted SSD drive in the PC

#======================================================================================================================
# Select this path  for training data
#FEAT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Training\Vertigo")

# Select this path  for testing data
FEAT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Testing\Vertigo")

# Raw Data Directory (To extract the 'round_end' ground truth)
RAW_DATA_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\datasets\billpureskillgg\cs2-2023-11-23\versions\2\csds")

# Standardized Team Mapping
TEAM_MAP = {"2": "T", "3": "CT", "TERRORIST": "T", "CT": "CT", "T": "T"}

# ==========================================
# 2. DATA RETRIEVAL & UPDATING LOGIC
# ==========================================
def update_features_with_targets():
    """Iterates through feature CSVs, retrieves ground-truth labels, and appends them."""
    
    if not FEAT_DIR.exists():
        print(f"Error: Feature directory does not exist -> {FEAT_DIR}")
        return

    csv_files = list(FEAT_DIR.glob("*.csv"))
    if not csv_files:
        print(f"No CSV files found in -> {FEAT_DIR}")
        return

    print(f"Found {len(csv_files)} matches to update. Processing...\n")
    updated_count = 0

    for filepath in csv_files:
        match_id = filepath.stem
        
        # --- A. Find the Raw Round_End Data ---
        match_dirs = list(RAW_DATA_DIR.rglob(match_id))
        if not match_dirs:
            print(f"Skipping {match_id}: Raw data directory not found.")
            continue
            
        re_files = glob.glob(str(match_dirs[0] / "round_end*"))
        if not re_files:
            print(f"Skipping {match_id}: 'round_end' channel missing.")
            continue

        # --- B. Extract the Target Labels ---
        try:
            df_re = pl.read_parquet(re_files[0])
            
            # Map the winner codes and keep only 'round' and 'target'
            df_labels = df_re.with_columns(
                pl.col("winner_team_code").cast(pl.Utf8).replace(TEAM_MAP).alias("target")
            ).select(["round", "target"])
            
        except Exception as e:
            print(f"Error reading 'round_end' for {match_id}: {e}")
            continue

        # --- C. Update the Feature Data ---
        try:
            df_feat = pl.read_csv(filepath)
            
            # Safety check: If "target" already exists from a previous run, drop it to prevent duplicates like "target_right"
            if "target" in df_feat.columns:
                df_feat = df_feat.drop("target")
                
            # Perform a Left Join on the "round" column
            # This ensures every round in the feature data gets its corresponding label
            df_updated = df_feat.join(df_labels, on="round", how="left")
            
            # --- D. Save the Changes (Overwrite) ---
            df_updated.write_csv(filepath)
            updated_count += 1
            
            print(f"Successfully updated: {match_id}")
            
        except Exception as e:
            print(f"Error updating feature file {match_id}: {e}")

    print("\n" + "="*40)
    print(f" UPDATE COMPLETE: {updated_count}/{len(csv_files)} files modified.")
    print("="*40)

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # Note: Because this script overwrites data, ensure no other processes are locking the CSV files.
    update_features_with_targets()

Found 40 matches to update. Processing...

Successfully updated: a8da3b6e-5aa6-4491-b3bb-6d7ad2cde52d
Successfully updated: 4e215d33-8702-4306-9cdb-6626ed93008c
Successfully updated: eb093ecf-1ef7-48f4-89a9-8e7f304d392a
Successfully updated: f08fecf8-0326-4248-8fde-c31ea4674cab
Successfully updated: 54470be9-4051-4e8d-bd4d-575378d625c9
Successfully updated: bb7b7fe9-4a91-4837-8cf3-74e845300d22
Successfully updated: 63ada926-89f0-4265-b5b0-07672dedd1df
Successfully updated: 18fe32ca-703d-4f86-b1ed-acb1e9b383fb
Successfully updated: d52a824c-4f82-4855-8181-de8a41d6baa4
Successfully updated: a57d811e-16d6-46a4-9b68-46d726423ce3
Successfully updated: 9a5712bc-ee4b-4241-86d3-32ce1cb15e67
Successfully updated: e11f10c9-e63c-45a3-ae2b-6f819e170dbf
Successfully updated: 6dd54cb2-8fc4-464f-aca4-7ce779781a20
Successfully updated: 870b89d7-1499-4626-9645-f8b062fa3841
Successfully updated: 5699bcb4-f1a3-4314-aaa6-e5038824a963
Successfully updated: e4dcb4d7-7402-4464-afd5-ec8df9d10004
Successfully 